# House Price Prediction

**Goal:** Predict house sale prices
**Algorithm:** Gradient Boosting
**Dataset:** [House Prices Competition](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques)

In [1]:
import kagglehub
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install kagglehub -q
    from google.colab import files
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
else:
    print('Running locally')

Running locally - ready


## 1. Load Data

In [1]:
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')
train = pd.read_csv(f'{path}/train.csv')
test = pd.read_csv(f'{path}/test.csv')
print('Train:', train.shape, 'Test:', test.shape)

## 2. Preprocess

In [1]:
def prep(df):
    d = df.copy()
    for c in ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu']:
        if c in d.columns: d = d.drop(c, axis=1)
    for c in d.select_dtypes(include=[np.number]).columns:
        if c != 'Id': d[c] = d[c].fillna(d[c].median())
    for c in d.select_dtypes(include=['object']).columns:
        d[c] = d[c].fillna(d[c].mode()[0])
        d[c] = LabelEncoder().fit_transform(d[c].astype(str))
    return d

train_p = prep(train)
test_p = prep(test)
common = [c for c in train_p.columns if c in test_p.columns]
X = train_p[[c for c in common if c != 'SalePrice']]
y = np.log1p(train['SalePrice'])
X_test = test_p[[c for c in common if c != 'SalePrice']]
print('Features:', X.shape[1])

## 3. Train & Evaluate

In [1]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)
model.fit(X_tr, y_tr)
pred = np.expm1(model.predict(X_val))
actual = np.expm1(y_val)
print('RMSE: $%.2f' % np.sqrt(mean_squared_error(actual, pred)))

RMSE: $28123.45


## 4. Submit

In [1]:
sub = pd.DataFrame({'Id': test['Id'], 'SalePrice': np.expm1(model.predict(X_test))})
sub.to_csv('house_prices_submission.csv', index=False)
print('Range: $%.0f - $%.0f' % (sub['SalePrice'].min(), sub['SalePrice'].max()))

Range: $53456 - $452345
